In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
import joblib

# Импортируем наш препроцессор
from src import create_preprocessor

print("Импорты выполнены")

Импорты выполнены


In [2]:
# Загружаем данные
df = pd.read_csv('../data/train.csv')
X = df.drop('Survived', axis=1)
y = df['Survived']

# Разделяем (стратифицированно)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

Train size: (712, 11), Test size: (179, 11)


In [3]:
preprocessor = create_preprocessor()

# Обучаем на train и преобразуем train и test
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print(f"Размер после препроцессинга: train {X_train_prepared.shape}, test {X_test_prepared.shape}")

Размер после препроцессинга: train (712, 13), test (179, 13)


In [5]:
# Модель 1: Logistic Regression
logreg_params = {
    'C': [0.1, 1.0, 10.0],
    'penalty': ['l2'],
    'solver': ['lbfgs']
}

# Модель 2: Random Forest
rf_params = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5, None],
    'min_samples_split': [2, 5]
}

# Модель 3: XGBoost
xgb_params = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0]
}

# Словарь моделей для удобства
models = {
    'LogisticRegression': (LogisticRegression(random_state=42, max_iter=1000), logreg_params),
    'RandomForest': (RandomForestClassifier(random_state=42), rf_params),
    'XGBoost': (XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False), xgb_params)
}

In [6]:
results = {}

for name, (model, params) in models.items():
    print(f"\n=== Обучаем {name} ===")
    
    # Создаем GridSearchCV
    grid = GridSearchCV(model, params, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)
    grid.fit(X_train_prepared, y_train)
    
    # Лучшие параметры
    best_model = grid.best_estimator_
    best_params = grid.best_params_
    best_cv_score = grid.best_score_
    
    # Предсказание на тесте
    y_pred = best_model.predict(X_test_prepared)
    y_pred_proba = best_model.predict_proba(X_test_prepared)[:, 1]
    test_roc_auc = roc_auc_score(y_test, y_pred_proba)
    test_accuracy = accuracy_score(y_test, y_pred)
    
    # Сохраняем результаты
    results[name] = {
        'best_params': best_params,
        'cv_roc_auc': best_cv_score,
        'test_roc_auc': test_roc_auc,
        'test_accuracy': test_accuracy,
        'model': best_model
    }
    
    print(f"Лучшие параметры: {best_params}")
    print(f"CV ROC-AUC: {best_cv_score:.4f}")
    print(f"Test ROC-AUC: {test_roc_auc:.4f}")
    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(classification_report(y_test, y_pred))


=== Обучаем LogisticRegression ===
Fitting 5 folds for each of 3 candidates, totalling 15 fits


C:\Users\Game_PC\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Лучшие параметры: {'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs'}
CV ROC-AUC: 0.8556
Test ROC-AUC: 0.8482
Test Accuracy: 0.8156
              precision    recall  f1-score   support

           0       0.82      0.89      0.86       110
           1       0.80      0.70      0.74        69

    accuracy                           0.82       179
   macro avg       0.81      0.79      0.80       179
weighted avg       0.81      0.82      0.81       179


=== Обучаем RandomForest ===
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Лучшие параметры: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}
CV ROC-AUC: 0.8716
Test ROC-AUC: 0.8419
Test Accuracy: 0.8156
              precision    recall  f1-score   support

           0       0.80      0.93      0.86       110
           1       0.85      0.64      0.73        69

    accuracy                           0.82       179
   macro avg       0.82      0.78      0.79       179
weighted avg       0.82      0.82      0.

C:\Users\Game_PC\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:01:22] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [7]:
# Создаем DataFrame для сравнения
comparison = pd.DataFrame({
    'Model': list(results.keys()),
    'CV ROC-AUC': [results[m]['cv_roc_auc'] for m in results],
    'Test ROC-AUC': [results[m]['test_roc_auc'] for m in results],
    'Test Accuracy': [results[m]['test_accuracy'] for m in results]
})

print("\n=== Сравнение моделей ===")
print(comparison.to_string(index=False))


=== Сравнение моделей ===
             Model  CV ROC-AUC  Test ROC-AUC  Test Accuracy
LogisticRegression    0.855593      0.848221       0.815642
      RandomForest    0.871646      0.841897       0.815642
           XGBoost    0.878959      0.834124       0.798883


In [8]:
# Выбираем лучшую по Test ROC-AUC
best_model_name = max(results, key=lambda x: results[x]['test_roc_auc'])
best_model = results[best_model_name]['model']

# Создаём папку models, если её нет
os.makedirs('../models', exist_ok=True)

# Сохраняем модель и препроцессор
joblib.dump(best_model, f'../models/best_model_{best_model_name}.pkl')
joblib.dump(preprocessor, '../models/preprocessor.pkl')

print(f"Сохранены: best_model_{best_model_name}.pkl и preprocessor.pkl")

Сохранены: best_model_LogisticRegression.pkl и preprocessor.pkl


In [9]:
loaded_model = joblib.load(f'../models/best_model_{best_model_name}.pkl')
loaded_preprocessor = joblib.load('../models/preprocessor.pkl')

# Проверяем на тестовой выборке (нужно применить препроцессор)
X_test_prepared_loaded = loaded_preprocessor.transform(X_test)
y_pred_loaded = loaded_model.predict(X_test_prepared_loaded)

print("ROC-AUC после загрузки:", roc_auc_score(y_test, loaded_model.predict_proba(X_test_prepared_loaded)[:, 1]))

ROC-AUC после загрузки: 0.8482213438735178


In [4]:
from src.train import get_models_and_params, train_and_select_model, save_model_and_preprocessor

# ... после подготовки X_train_prepared, X_test_prepared ...
best_model, results, best_name = train_and_select_model(X_train_prepared, y_train, X_test_prepared, y_test)
save_model_and_preprocessor(best_model, preprocessor, best_name)


=== Training LogisticRegression ===
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Test ROC-AUC: 0.8482

=== Training RandomForest ===
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Test ROC-AUC: 0.8419

=== Training XGBoost ===
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Test ROC-AUC: 0.8341
Saved model and preprocessor to ../models/


C:\Users\Game_PC\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:09:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
